# Environment
For this project I will use the following libraries:
- `jupyter` to use jupyter notebooks
- `requests` to fecth the knowledge base
- `sentence-transformers` for embedding
- `minsearch` to index the knowlede base and provide a search functionality
- `python-dotenv` to load environmental variables
- `openai` to communicate with the LLM

## Variable Reference (Data Dictionary)

Quick map of the main variables in this project.  

**The golden rule:** position `i` always refers to the same FAQ across lists and arrays:  
`documents[i]`, `faq_texts[i]`, `vectors[i]`, `faq_embeddings[i]`, and `similarity_scores[i]` all describe the same item.

---

### 1. Raw data (text, not vectors)

| Variable | Type | Description |
| :--- | :--- | :--- |
| `courses_raw` | `list[dict]` | Course catalog from the DataTalks.Club API. One entry per Zoomcamp course (name, path, etc.). Not used directly for search — only to fetch FAQs. |
| `documents` | `list[dict]` | **The knowledge base.** All FAQ records loaded via `load_faq_data()`. Each dict has: `course`, `section`, `FAQ` (question), `answer`, `doc_id`. This is what you read when you want the actual text of a result. |
| `df` | `pandas.DataFrame` | Tabular view of `documents` for inspection only. Same data, easier to browse in the notebook. |

**Example:** `documents[2]` → the FAQ dict at position 2 (question + answer + metadata).

---

### 2. Text prepared for embedding

| Variable | Type | Description |
| :--- | :--- | :--- |
| `faq_texts` | `list[str]` | One string per FAQ, built as `"<FAQ> Answer: <answer>"`. This is the exact text sent to the embedding model. Same length and order as `documents`. |

**Example:** `faq_texts[2]` → the combined question+answer string for `documents[2]`.

---

### 3. Embedding model

| Variable | Type | Description |
| :--- | :--- | :--- |
| `embedder` | `SentenceTransformer` | The loaded model (`all-MiniLM-L6-v2`). Converts any text string into a 384-dimensional vector. Call `embedder.encode(text)` to embed one string or a list of strings. |

---

### 4. Document embeddings (vectors, not readable text)

| Variable | Type | Description |
| :--- | :--- | :--- |
| `vectors` | `list` | Raw list of embedding vectors (one per FAQ), built in batches. Intermediate step before converting to NumPy. |
| `faq_embeddings` | `np.ndarray` shape `(N, 384)` | **All FAQ vectors in one matrix.** Row `i` is the embedding of `faq_texts[i]` / `documents[i]`. Used for fast similarity search against a query. You cannot read FAQ text from this — only numbers. |
| `batch_size` | `int` | How many texts to embed per batch (50). Memory/performance setting, not chunking. |

---

### 5. Query (user question)

| Variable | Type | Description |
| :--- | :--- | :--- |
| `query` | `str` | The user's natural-language question (plain text). |
| `v_user_query` | `np.ndarray` shape `(384,)` | The embedding of `query`. Compared against every row of `faq_embeddings` to find relevant FAQs. |

---

### 6. Similarity scores (numbers, not documents)

| Variable | Type | Description |
| :--- | :--- | :--- |
| `similarity_scores` | `np.ndarray` shape `(N,)` | One score per FAQ — dot product between `v_user_query` and each row of `faq_embeddings`. Higher = more similar. With this model, scores are cosine similarity (roughly 0 to 1). **Use this for sorting/ranking, not for reading content.** |
| `relevancy_scores` | `np.ndarray` shape `(N,)` | Same as `similarity_scores`, computed with a slow Python loop (for teaching). Should match `similarity_scores` via `np.allclose`. |
| `dot_product` | `list` | Temporary list used while building `relevancy_scores` in the loop. |

---

### 7. Indices & results (connecting scores → documents)

| Variable | Type | Description |
| :--- | :--- | :--- |
| `top_match_index` | `int` | Position of the single best-matching FAQ in `similarity_scores` (`np.argmax`). |
| `asc_docs_indices` | `np.ndarray` | All FAQ positions sorted from **lowest** to **highest** similarity score. |
| `desc_docs_indices` | `np.ndarray` | All FAQ positions sorted from **highest** to **lowest** similarity score. |
| `top_5_indices` | `np.ndarray` shape `(5,)` | Positions of the top 5 most relevant FAQs (best first if using `np.argsort(-similarity_scores)[:5]`). |
| `documents[i]` | `dict` | **How you get the actual FAQ text** once you have an index `i` from any of the variables above. |

**Example flow:**
```python
top_5_indices = np.argsort(-similarity_scores)[:5]
top_5_docs = [documents[i] for i in top_5_indices]
```

---

### 8. Tutorial-only variables (embedding demo)

Used earlier to explain dot products with hardcoded examples. Not part of the full-dataset search pipeline.

| Variable | Description |
| :--- | :--- |
| `q_1`, `q_2` | Sample user questions for the tutorial |
| `v_1`, `v_2` | Embeddings of `q_1`, `q_2` |
| `doc` | A hardcoded answer string (not from `documents`) |
| `v_doc` | Embedding of that hardcoded `doc` |

---

### Cheat sheet: which variable do I use?

| I want to… | Use |
| :--- | :--- |
| Read FAQ question/answer text | `documents` |
| Embed new text | `embedder.encode(...)` |
| Compare query to all FAQs (fast) | `faq_embeddings.dot(v_user_query)` → `similarity_scores` |
| Sort FAQs by relevance | `np.argsort(-similarity_scores)` |
| Get top 5 FAQ dicts | `[documents[i] for i in top_5_indices]` |
| Get top 5 scores | `similarity_scores[top_5_indices]` |

In [188]:
import requests

docs_url = 'https://datatalks.club/faq/json/courses.json'
response = requests.get(docs_url)
# Return a list of the courses offered by the DataTalks.Club -> list(dict)
courses_raw = response.json()

In [189]:
print(f"Number of courses offered: {len(courses_raw)}")

Number of courses offered: 6


## Retrieval
Let's retrieve all the FAQs for each course.

In [190]:
from ingest import load_faq_data

documents = load_faq_data()
print(documents[0])

{'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel.", 'doc_id': '9e508f2212', 'FAQ': 'Course: When does the course start?'}


In [191]:
# Use pandas for cosmetic purposes (visualizing the data)
import pandas as pd

df = pd.DataFrame(documents)
df.head(7)

,course,section,answer,doc_id,FAQ
0,data-engineering-zoomcamp,General Course-Related Questions,A new cohort runs roughly January–April every ...,9e508f2212,Course: When does the course start?
1,data-engineering-zoomcamp,General Course-Related Questions,"To get the most out of this course, you should...",bfafa427b3,Course: What are the prerequisites for this co...
2,data-engineering-zoomcamp,General Course-Related Questions,"Yes, even if you don't register, you're still ...",3f1424af17,Course: Can I still join the course after the ...
3,data-engineering-zoomcamp,General Course-Related Questions,You don't need a confirmation email. You're ac...,52217fc51b,Course: I have registered for the Data Enginee...
4,data-engineering-zoomcamp,General Course-Related Questions,Get the basic environment ready ahead of time:...,33fc260cd8,Course: What can I do before the course starts?
5,data-engineering-zoomcamp,General Course-Related Questions,DataTalks.Club runs several Zoomcamps every ye...,b71fb3b195,Course: how many Zoomcamps run in a year?
6,data-engineering-zoomcamp,General Course-Related Questions,Most of the syllabus stays consistent year to ...,e499535e82,Course: Is the current cohort going to be diff...


## Embedding the Knowledge Base
I will now use ***Sentence Transformers*** with the model all-MiniLM-L6-v2 to embed the documents of the knowledge base into a 384-dimensional vector space (the embedding dimension is the size of each vector, and is defined and fixed by the model).

In [192]:
# The first time, this import will download locaally the model from Hugging Face
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

### Testing how the embedding works

In [193]:
# User poses a question
q_1 = "Can I still join the course after the start date?"

# Embed the user query into a vector
v_1= embedder.encode(q_1)

# Print the shape of the embedding
print(f"The shape of the embedding is: {v_1.shape}")

The shape of the embedding is: (384,)


As we see, the model `all-MiniLM-L6-v2`, which uses a ***384-dimension vector space***, produced a one-dimensional array of length `384`.  

Below, I will hardcode the answer from the knowledge base to show how we can use **embedding**.

In [194]:
# Hard coding document's answer from the knowledge base assuming this was returned by the search
doc = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."

# Embed the document's answer
v_doc = embedder.encode(doc)

# Print the shape of the embedding
print(f"The shape of the embedding is: {v_doc.shape}")

The shape of the embedding is: (384,)


#### Comparing the vectors
We compare the user query against the retrieved document using dot product.  


> **NOTE: The dot product is commutative only when multiplying 1D Vectors**  
>
>Let $A$ and $B$ be two ***3D vectors***:  
>
>$\mathbf{A} = [2, -3, 4]$
>$\mathbf{B} = [5, 1, 2]$  
>
>$\mathbf{A}\,dot\,\mathbf{B}$  
>>$\mathbf{A} \cdot \mathbf{B} = (2 \times 5) + (-3 \times 1) + (4 \times 2)$  
>>$\mathbf{A} \cdot \mathbf{B} = 10 - 3 + 8 = \mathbf{15}$  
>
>$\mathbf{B}\,dot\,\mathbf{A}$   
>>$\mathbf{B} \cdot \mathbf{A} = (5 \times 2) + (1 \times -3) + (2 \times 4)$  
>>$\mathbf{B} \cdot \mathbf{A} = 10 - 3 + 8 = \mathbf{15}$


Let's run the comaparison using the `dot` method provided by Numpy (`v_1` and `v_doc` are numpy array returned the emebedding).
>**NOTE:** When a vector is normalized, the `dot` product is the same as Cosine Similarity.  
>
> - Because `all-MiniLM-L6-v2` outputs normalized vectors, the **dot-product** between any two vectors **is** the **cosine similarity**. 

Therefore the range of the dot-product is `[-1,1]`.

- $-1$ indicating completely opposite|
- $0$ indicating no relation, neither opposite nor similar|
- $1$ indicating 100% similarity (basically if the cosine similarity is 1 we have the vectors pointing in the same direction)  

In practice, we rarely get cosine similarity below 0. The embedding model maps text to a region of the vector space where most vectors have positive components.   
There's no concept of "opposite meaning" that maps to a vector pointing the other way.

In [195]:
# The dot product is commutative, so it doesn't matter which vector we use first
print(f"Query • Document = {v_1.dot(v_doc)}")
print(f"Document • Query = {v_doc.dot(v_1)}")

Query • Document = 0.3233240246772766
Document • Query = 0.3233240246772766


Now with an unrelated query:

In [196]:
q_2 = "How to install Docker on Windows?"
v_2 = embedder.encode(q_2)

In [197]:
print(f"Document • Query = {v_doc.dot(v_2)}")

Document • Query = 0.01973051019012928


The first score for `q_1` vs `doc` (`0.32`) is higher, so that query is more similar to the document about registration.  
The second score for `q_2` vs `doc` sits near `0`, because installing Docker has nothing to do with registration.  

A score near `0` means the two vectors are about as different as they can be.

### Embedding Our actual Dataset  

Each document is a Python dictionary with a question (FAQ) and an answer.  
We embed both together, that way a query can match both the question text and the answer text in our index.

In [198]:
documents[:2]

[{'course': 'data-engineering-zoomcamp',
  'section': 'General Course-Related Questions',
  'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel.",
  'doc_id': '9e508f2212',
  'FAQ': 'Course: When does the course start?'},
 {'course': 'data-engineering-zoomcamp',
  'section': 'General Course-Related Questions',
  'answer': "To get the most out of this course, you should have:\n\n- Basic coding experience\n- Familiarity with SQL\n- Experience with Python (helpful but not required)\n\nNo prior data engineering experience is necessary. See [Readme on GitHu

In [199]:
# Let's build one string per document, with the question and the answer 
faq_texts = [f"{doc['FAQ']} Answer: {doc['answer']}" for doc in documents]


print(f"The are {len(faq_texts)} FAQs with their answers.")

The are 1350 FAQs with their answers.


#### Using `tqdm` library to show visual progress  

`tqdm` is a progress bar library. It wraps any loop and shows you a live progress bar while the loop runs and how many items have been processed, how many are left, and estimated time to completion.  

Without tqdm the loop runs silently, we have no idea how far along it is.

With tqdm we get a progress bar like this:  
```zsh
45%|████████          | 450/1000 [00:12<00:15, 32.4it/s]
```

The next step is embedding a large number of documents (~1300), which takes time. `tqdm` just makes that loop visible so we will not just staring at a frozen screen.

In [200]:
from tqdm.auto import tqdm

### Embed the dataset in batches
Process all documents through the embedding model in groups of 50. This is batch processing — a memory and efficiency optimization — not document chunking.  

The document count stays the same: N documents in, N vectors out.

In [201]:
batch_size = 50
vectors = []

# tqdm is a method that shows a progress bar
for i in tqdm(range(0, len(faq_texts), batch_size)):
    # Get the current batch of texts
    batch = faq_texts[i:i+batch_size]
    # Embed the current batch
    current_batch_vectors = embedder.encode(batch)
    # Save current batch's vectors by adding them to the list
    vectors.extend(current_batch_vectors)

print(f"Done! We have {len(vectors)} vectors.")

  0%|          | 0/27 [00:00<?, ?it/s]

Done! We have 1350 vectors.


#### Converting `vectors` to a Matrix
Now that we have a list of vectors, which basically is a `list[lists]`,
we need to convert it to a matrix. 

A `list[lists]` and a "NumPy 2D Array" look similar, but they act very differently under the hood.

We'll convert the `list[lists]` to a NumPy 2D-array to move from Python-land (slow) to C-land (fast) (Numpy is implemenrted in `C` language).

##### Why NumPy and not a Python's `list[lists]`?
1. **Speed**. To find the most relevant `FAQ-Answer` pair (a vector) to a user query, we need to compute the dot product between the user query vector and every FAQ-Answer document vector. With a python list of lists we would need to loop through each document one by one -> very slow. With a NumPy matrix we do it in a single operation

2. **Math operations**. NumPy matrices (2D-Array) support vector math directly. A list of lists is just a Python data structure that has no concept of dot products, matrix multiplication, or similarity scores.

The choice for using NumPy is specifically so that the search steps (matching a user query vector to all FAQ-Answer document vectors) can be done as a single fast matrix operation instead of a slow Python loop.

In [202]:
import numpy as np

In [203]:
faq_embeddings = np.array(vectors)
print(f"The shape of the matrix is: {faq_embeddings.shape}.")
print(f"{faq_embeddings.shape[0]} vectors with {faq_embeddings.shape[1]} dimensions (descriptive features.")

The shape of the matrix is: (1350, 384).
1350 vectors with 384 dimensions (descriptive features.


### Vector Search

Vector search works like so: when a **user query** arrives, we **embed** it and then we compare it against every (embedded) document, and keep the most similar (relevant) ones.



In [204]:
query = "Can I still join the course after the start date?"
# Embed the query into a vector
v_user_query = embedder.encode(query)

# Print the shape of the embedding
print(f"The shape of the query embedding is: {v_user_query.shape}")

The shape of the query embedding is: (384,)


#### Search against the embdedded knowledge base
Next, we just compute the `dot-prodcut` of the entire matrix with the embedded query.  

>**Note**:  
The *dot-product* betyween two vectors, **in any dimensions**, is only possible if:
- If the last dimension of the first vector = to the second last dimension of the second vector  

| Operation | Result | Notes |
| :--- | :--- | :--- |
| $(3, 4) \cdot (4, 12)$ | $(3, 12)$ | Valid matrix multiplication $(m \times n) \cdot (n \times p) = (m \times p)$ |
| $(4, 12) \cdot (3, 4)$ | $\text{NOT POSSIBLE}$ | Inner dimensions do not match ($12 \neq 3$) |
| $(1, 4, 5, 6) \cdot (4, 5, 8, 12, 15, 6, 34)$ | $(1, 4, 5, 4, 5, 8, 12, 15, 34)$ | Represents a vector-matrix product or alignment check |
| $(3, ) \cdot (3, 9)$ | $(1, 9)$ | When $(3,)$ is the left operand, NumPy expands it to $(1, 3)$ |
| $(3, 9) \cdot (9, )$ | $(3, 1)$ | When $(9,)$ is the right operand, NumPy expands it to $(9, 1)$ |
| $(5, ) \cdot (3, 4)$ | $\text{NOT POSSIBLE}$ | Promotion to $(1, 5) \cdot (3, 4)$ fails inner dimension check |


In [205]:
# Veryfing the shapes to perform the dot product in the correct order
# Dot product is not commutative when multiplying 2D+ Vectors
print(f"{faq_embeddings.shape} * {v_user_query.shape}")

similarity_scores = faq_embeddings.dot(v_user_query)
print(f"The shape of the ranked results is: {similarity_scores.shape}")
similarity_scores

(1350, 384) * (384,)
The shape of the ranked results is: (1350,)


array([ 0.4816382 ,  0.19906138,  0.77913874, ..., -0.09574807,
        0.03110426, -0.03260853], shape=(1350,), dtype=float32)

#### Note on dot multiplication

When we multiply `faq_embeddings` with `v_user_query` the respective shapes are `(1350,384)` and `(384,)`.  

`faq_embeddings` is a 2D vector, while `v_user_query` is a 1D vector. When numpy performs the dot product, what it does behind the scenes, is several multiplication of two 1D vectors. Why?  

Because each "row" of `faq_embeddings` is a 1D vectors (you can view `faq_embeddings` as vectorized list of lists). So it runs a loop which takes each of the 1350 1D vectors (each have 384 elements) and multiply it by the `v_user_query` 1D vector (that has 384 elements, so dimensions checks out).  

We could implement that with a python loop but it would be slower and incovenient:

In [206]:
dot_product = []
for v in faq_embeddings:
    product = v.dot(v_user_query)
    dot_product.append(product)

relevancy_scores = np.array(dot_product)

In [207]:
# Check if the two arrays are the same for practical purposes
np.allclose(relevancy_scores, similarity_scores) # True

True

#### Best match  
`np.argmax(np.Array)` returns the **index** of the highest value in an array, **not the value itself**, but **its position**.

In [208]:
# Get the index of the highest value (cosine similarity) in the ranked_faqs array
top_match_index = np.argmax(similarity_scores)
top_match_index

np.int64(2)

In [209]:
top_match_index, similarity_scores[top_match_index]

(np.int64(2), np.float32(0.77913874))

In [210]:
# Get the document ranked as best match for the user query
documents[top_match_index]


{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.",
 'doc_id': '3f1424af17',
 'FAQ': 'Course: Can I still join the course after the start date?'}

#### Top N Results  
`np.argsort(np.Array)` returns the **indices** that would sort the array in ascending order, **not the sorted values**, but their position.


In [211]:
# Indices of the documents sorted in ascending order
asc_docs_indices = np.argsort(similarity_scores)
desc_docs_indices = np.argsort(similarity_scores)[::-1]

print(f"The ascending documents indices are: {asc_docs_indices}")
print(f"The descending documents indices are: {desc_docs_indices}")


The ascending documents indices are: [ 217  964 1089 ...  907  625    2]
The descending documents indices are: [   2  625  907 ... 1089  964  217]


In [212]:
top_5_indices = np.argsort(similarity_scores)[-5:]

A trick to get the array ordered in descent order is to the negate the `similarity_scores` argument first:

In [213]:
# Indice of the top 5 documents in descending order (best to "worst")
top_5_indices = np.argsort(-similarity_scores)[:5]


>**NOTE**: The answer to a question can be spread across several documents. One holds part of it, another fills in the rest. Sometimes the top result isn't the right one but the second is. **We send all 5 to the LLM and let it combine them.**

Let's the the match of the top 5 most relevant document to the user query.  
As a reminder the user query was: "***Can I still join the course after the start date?***".  

In [214]:
for i in top_5_indices:
    print(f"Document {i}: {documents[i]}")
    print(f"Similarity score: {similarity_scores[i]}")
    print("-"*100)


Document 2: {'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.", 'doc_id': '3f1424af17', 'FAQ': 'Course: Can I still join the course after the start date?'}
Similarity score: 0.7791387438774109
----------------------------------------------------------------------------------------------------
Document 625: {'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute.", 'doc_id': '2d8b16c2a0', 'FAQ': 'Course - Can I still join

## Implementing the Vector Search with `minsearch`  

So far we did vector search manually:  

1. embedded the user query -> `v_user_query`
2. calculated dot-product (cosine similarity) between the `faq_embeddings` and the `v_user_query`  
3. retrieved the top 5 indices of the best cosine similarities from `similarty_scores`  
4. retrieved the top 5 documents (faqs) mapped by the top 5 indeces above  

### The Issues with this approach
1. **Repetitive**: for every user query we receive we need to do this manually writing the same sequence of code.
2. **No Filtering**: No buil-in way to filter the search for a specific course. For every query, the entire knwoledge base is searched, even for irrelevant courses for the query.  

### What `minsearch` adds  
- `VectorSearch` wraps all of that into a clean two-steps process.
- Filering is built in

We already have the documents and vectors from above.

- `faq_texts`: plain text of the knowledge base (all courses' faqs)
- `faq_embeddings`: the embedded knowledge base as a matrix of all courses' faqs.  
Matrix shape:(N_faqs, fix_embedding_length) 

In [215]:
from minsearch import VectorSearch

In [216]:
# Create a empty minsearch index adding that the course field can be used for filtering
v_index = VectorSearch(keyword_fields=["course"])

In [217]:
# Indexing the vectorized knowledge (numbers) base with the actual documents (text)
# This allows to search semantically the knowledge base while also filtering by course
v_index.fit(faq_embeddings, documents)

#### Why `fit()` takes both the `faq_embeddings` and `documents`?

>**NB**: `minsearch` requires an iterable for the text payload, so we use the retrieved documents (`list[dict]`)

`v_index.fit(faq_embeddings, documents)` stores two parallel lists internally:

- `faq_embeddings`: the vectors it searches through mathematically
- `documents`: the FAQ text it returns to you after search

Position `i` in `faq_embeddings` always maps to position `i` in `documents`:

`faq_embeddings[0]` ↔ `documents[0]` → `{"question": "Can I join late?", ...}`

Neither is useful alone.  
- `faq_embeddings` without `documents` gives you an index position that means nothing.  
- `documents` without `faq_embeddings` gives you text with nothing to search through.  
- `fit()` links them so `search()` can score against `faq_embeddings` and return the matching entry from `documents` in one call.

### Performing a search

In [218]:
query = "I just discovered the course. Can I still join it?"
query_vector = embedder.encode(query)

results = v_index.search(query_vector, num_results=5)

In [219]:
# Top results
results[0]


{'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'doc_id': '74eb249bbf',
 'FAQ': 'I just discovered the course. Can I still join?'}

### Filtering by course

In [220]:
results = v_index.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

## RAG with Vector Search  

Continue to notebook `notebook_RAG.ipynb`